In [ ]:
import scanpy as sc

import matplotlib.pyplot as plt
import loompy

两个小坑，1。umap图为什么不是paga的 2.

In [ ]:
adata

In [ ]:
import celloracle as co

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

In [ ]:
import celloracle as co
co.__version__

In [ ]:
# visualization settings
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

In [ ]:
save_folder = "figures"
os.makedirs(save_folder, exist_ok=True)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=2000)

# 查看结果
print(f"共有 {adata.var['highly_variable'].sum()} 个高变基因")

# 筛选出高变基因子集（可选）
adata_hvg = adata[:, adata.var['highly_variable']].copy()

In [ ]:
hvg_list = adata.var_names[adata.var['highly_variable']].tolist()

In [ ]:
import numpy as np

# 你想强制加入的基因列表
extra_genes = ["Sox10", "Phox2b", "Zeb2", "Ascl1", "Hand2"]

# 如果这些基因在 adata.var_names 中存在，就把它们的 highly_variable 设为 True
for g in extra_genes:
    if g in adata.var_names:
        adata.var.loc[g, "highly_variable"] = True
    else:
        print(f"⚠️ 基因 {g} 不在数据中，已跳过")

# 检查新的高变基因数量
print(f"新的高变基因总数：{adata.var['highly_variable'].sum()}")

# 可选：提取更新后的高变基因子集
adata_hvg = adata[:, adata.var['highly_variable']].copy()

In [ ]:
print(f"Cell number is :{adata_hvg.shape[0]}")
print(f"Gene number is :{adata_hvg.shape[1]}")

In [ ]:
adata=adata_hvg

In [ ]:
# Load TF info which was made from mouse cell atlas dataset.
base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()

# Check data
base_GRN.head()

In [ ]:
oracle = co.Oracle()

In [ ]:
# Check data in anndata
print("Metadata columns :", list(adata.obs.columns))
print("Dimensional reduction: ", list(adata.obsm.keys()))

In [ ]:


# Instantiate Oracle object.
oracle.import_anndata_as_raw_count(adata=adata,
                                   cluster_column_name="cell_subtype",
                                   embedding_name="X_umap")

In [ ]:
# You can load TF info dataframe with the following code.
oracle.import_TF_data(TF_info_matrix=base_GRN)

# Alternatively, if you saved the informmation as a dictionary, you can use the code below.
# oracle.import_TF_data(TFdict=TFinfo_dictionary)

In [ ]:
Paul_15_data = Paul_15_data.rename(columns={
    'reg_pbmc.TF': 'TF',
    'FTILER': 'Target_genes',
    # 可以继续添加需要修改的列名
})
Paul_15_data

In [ ]:
# Make dictionary: dictionary key is TF and dictionary value is list of target genes.
TF_to_TG_dictionary = {}

for TF, TGs in zip(Paul_15_data.TF, Paul_15_data.Target_genes):
    # convert target gene to list
    TG_list = TGs.replace("", "").split(",")
    # store target gene list in a dictionary
    TF_to_TG_dictionary[TF] = TG_list

# We invert the dictionary above using a utility function in celloracle.
TG_to_TF_dictionary = co.utility.inverse_dictionary(TF_to_TG_dictionary)

In [ ]:
# Add TF information
oracle.addTFinfo_dictionary(TG_to_TF_dictionary)

In [ ]:
# Perform PCA
oracle.perform_PCA()

# Select important PCs
plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
plt.axvline(n_comps, c="k")
plt.show()
print(n_comps)
n_comps = min(n_comps, 50)

In [ ]:
n_cell = oracle.adata.shape[0]
print(f"cell number is :{n_cell}")


In [ ]:
k = int(0.025*n_cell)
print(f"Auto-selected k is :{k}")

In [ ]:
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8,
                      b_maxl=k*4, n_jobs=4)

In [ ]:
# Save oracle object.
#oracle.to_hdf5("Paul_15_data.celloracle.oracle")

In [ ]:
# Load file.
#oracle = co.load_hdf5("Paul_15_data.celloracle.oracle")

In [ ]:
# Check clustering data
sc.pl.umap(oracle.adata, color="cell_subtype")

In [ ]:
%%time
# Calculate GRN for each population in "louvain_annot" clustering unit.
# This step may take some time.(~30 minutes)
links = oracle.get_links(cluster_name_for_GRN_unit="cell_subtype", alpha=10,
                         verbose_level=10)

In [ ]:
# Save Links object.
links.to_hdf5(file_path="links.celloracle.links")

In [ ]:
links.filter_links(p=0.001, weight="coef_abs", threshold_number=2000)

In [ ]:
plt.rcParams["figure.figsize"] = [9, 4.5]

In [ ]:
links.plot_degree_distributions(plot_model=True,
                                               #save=f"{save_folder}/degree_distribution/",
                                               )

In [ ]:
plt.rcParams["figure.figsize"] = [6, 4.5]

In [ ]:
# Calculate network scores.
links.get_network_score()

In [ ]:
links.merged_score.head()

In [ ]:
# Save Links object.
#links.to_hdf5(file_path="links.celloracle.links")
# You can load files with the following command.
#links = co.load_hdf5(file_path="links.celloracle.links")

In [ ]:
import os
import sys

import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns


In [ ]:
#plt.rcParams["font.family"] = "arial"
plt.rcParams["figure.figsize"] = [6,6]
%config InlineBackend.figure_format = 'retina'
plt.rcParams["savefig.dpi"] = 600

%matplotlib inline

In [ ]:
# Make folder to save plots
save_folder = "figures"
os.makedirs(save_folder, exist_ok=True)

In [ ]:
#oracle = co.load_hdf5("data/SCfate/Paul_15_data.celloracle.oracle")
#oracle

In [ ]:
#links = co.load_hdf5("data/SCfate/links.celloracle.links")
#links

In [ ]:
#links.filter_links()
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)
oracle.fit_GRN_for_simulation(alpha=10,
                              use_cluster_specific_TFdict=True)

In [ ]:
oracle._generate_meta_data

In [ ]:
# Check gene expression
goi = "Irf7"
sc.pl.umap(oracle.adata, color=[goi, oracle.cluster_column_name],
                 layer="imputed_count", use_raw=False, cmap="viridis")

In [ ]:
valid_tfs = list(oracle.all_regulatory_genes_in_TFdict)

In [ ]:
oracle.adata

In [ ]:
# Plot gene expression in histogram
sc.get.obs_df(oracle.adata, keys=[goi], layer="imputed_count").hist()
plt.show()

In [ ]:
oracle.simulate_shift(perturb_condition={goi: 0.0},
                      n_propagation=3)

In [ ]:
# Get transition probability
oracle.estimate_transition_prob(n_neighbors=200,
                                knn_random=True,
                                sampled_fraction=1)

# Calculate embedding
oracle.calculate_embedding_shift(sigma_corr=0.05)

In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale = 25
# Show quiver plot
oracle.plot_quiver(scale=scale, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_quiver_random(scale=scale, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()

In [ ]:
# n_grid = 40 is a good starting value.
n_grid = 40
oracle.calculate_p_mass(smooth=0.8, n_grid=n_grid, n_neighbors=200)

In [ ]:
# Search for best min_mass.
oracle.suggest_mass_thresholds(n_suggestion=12)

In [ ]:
min_mass = 13
oracle.calculate_mass_filter(min_mass=min_mass, plot=True)

In [ ]:
fig, ax = plt.subplots(1, 2,  figsize=[8, 3])

scale_simulation =20
# Show quiver plot
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector: {goi} KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_simulation_flow_random_on_grid(scale=scale_simulation, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()

In [ ]:
# Plot vector field with cell cluster
fig, ax = plt.subplots(figsize=[8, 8])
oracle.plot_cluster_whole(ax=ax, s=10)
oracle.plot_simulation_flow_on_grid(scale=scale_simulation, ax=ax, show_background=False)
ax.set_title(f"Simulated cell identity shift vector: {goi} KO")
plt.show()